In [0]:
%sql

select count(distinct idCliente) 

from workspace.tmw_loyalty.clientes

In [0]:
%sql
select 
    date(DtCriacao) as DtCriacao,
    count(distinct IdCliente) as QtdeClientes
from workspace.tmw_loyalty.transacoes
group by date(DtCriacao)
order by DtCriacao

In [0]:
%sql
select * from workspace.tmw_loyalty.transacoes
limit 10

In [0]:
%sql
select 
    date_trunc('month', DtCriacao) as Mes,
    count(distinct IdTransacao) as QtdeTransacoes
from workspace.tmw_loyalty.transacoes
group by date_trunc('month', DtCriacao)
order by Mes

In [0]:
%sql
select 
    date_trunc('year', DtCriacao) as Ano,
    count(distinct IdTransacao) as QtdeTransacoes
from workspace.tmw_loyalty.transacoes
group by date_trunc('year', DtCriacao)
order by Ano

Em 2026 quantos novos usuários?

In [0]:
%sql
select count(distinct idCliente) 

from workspace.tmw_loyalty.clientes
where year(date_trunc('year', DtCriacao)) = 2026

Qual produto mais transacionado? 

E o de maiores valores acumulados?

In [0]:
%sql
select 
    A.IDpRODUTO,
    b.DescNomeProduto,workspace.tmw_loyalty.transacao_produto
    count(distinct a.idTransacaoProduto) qtd_transcoes
from workspace.tmw_loyalty.transacao_produto A 
    left join workspace.tmw_loyalty.produtos b 
        on a.idproduto = b.IdProduto
group by all
order by 3 desc

In [0]:
%sql
select 
    A.IDpRODUTO,
    b.DescNomeProduto,
    sum(qtdeProduto) qtd
from workspace.tmw_loyalty.transacao_produto A 
    left join workspace.tmw_loyalty.produtos b 
        on a.idproduto = b.IdProduto
group by all
order by 3 desc

In [0]:
%sql
select * from workspace.tmw_loyalty.transacoes
limit 10 

In [0]:
%sql
select 
    date(DtCriacao) as DtCriacao,
    count(distinct IdCliente) as qtddetransacoes
from workspace.tmw_loyalty.transacoes
group by date(DtCriacao)
order by qtddetransacoes desc 

In [0]:
%sql
select 
    date(DtCriacao) as DtCriacao,
    count(distinct IdCliente) as QtdeClientes
from workspace.tmw_loyalty.clientes
group by date(DtCriacao)
order by QtdeClientes desc 

In [0]:
%sql
with tb_daily as 
(
SELECT
    DATE(DtCriacao) AS Data,
    IdCliente,
    sum(case when qtdePontos > 0 then QtdePontos else 0 end) as qtdePontos
FROM workspace.tmw_loyalty.transacoes
GROUP BY all
), 
tb_dau as 
(
select 
    data,
    count(distinct idcliente) dau
from tb_daily 
group by all
order by data
), tb_mau as 
(
select 
    t1.data, 
    t1.dau,
    count(distinct t2.idcliente) as mau,
    sum(qtdePontos) / count(distinct t2.idcliente) ARPU,
    count(distinct t2.data) as qtdeDias 
from tb_dau t1 
    left join tb_daily t2 
        on t1.data >= t2.data
            and t1.data - interval 28 days < t2.data

group by all
), tb_wau as 
(
select 
    t1.data, 
    t1.dau,
    t1.mau,
    t1.arpu,
    t1.qtdeDias,
    count(distinct t2.idcliente) as wau
from tb_mau t1 
    left join tb_daily t2 
        on t1.data >= t2.data
            and t1.data - interval 7 days < t2.data

group by all
)
select 
    *
from tb_wau
order by data 

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
with tb_daily as 
(
select distinct
    date(DtCriacao) as dtDia,
    IdCliente
from tmw_loyalty.transacoes
), tb_lag as 
(
select 
    *,
    lag(dtdia) over (partition by idcliente order by dtdia) as lagDt
from tb_daily
), tb_cliente as 
(
select 
    idcliente,
    avg(datediff(dtdia,lagdt)) dia_recorrencia
from tb_lag
where lagdt is not null
group by all
)
, tb_qtde as 
(
select 
    dia_recorrencia,
    count(distinct idcliente) qtd_clientes
from tb_cliente
group by all 
), tb_acum as 
(
select 
    *, 
    sum(qtd_clientes) over (order by dia_recorrencia) qtd_acum
from tb_qtde 
)
select *,
    qtd_acum / (select max(qtd_acum) from tb_acum) div_acum,
    1- div_acum curva_sobrev
from tb_acum

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
with tb_daily as 
(
SELECT
    DATE(DtCriacao) AS Data,
    IdCliente,
    sum(case when qtdePontos > 0 then QtdePontos else 0 end) as qtdePontos
FROM workspace.tmw_loyalty.transacoes
GROUP BY all
), 
tb_dau as 
(
select 
    data,
    count(distinct idcliente) dau
from tb_daily 
group by all
order by data
), tb_mau as 
(
select 
    t1.data, 
    t1.dau,
    count(distinct t2.idcliente) as mau,
    sum(qtdePontos) / count(distinct t2.idcliente) ARPU,
    count(distinct t2.data) as qtdeDias 
from tb_dau t1 
    left join tb_daily t2 
        on t1.data >= t2.data
            and t1.data - interval 28 days < t2.data

group by all
), tb_wau as 
(
select 
    t1.data, 
    t1.dau,
    t1.mau,
    t1.arpu,
    t1.qtdeDias,
    count(distinct t2.idcliente) as wau
from tb_mau t1 
    left join tb_daily t2 
        on t1.data >= t2.data
            and t1.data - interval 7 days < t2.data

group by all
)

, tb_mau_users as 
(
select distinct
    t1.data, 
    t2.idcliente
from tb_dau t1 
    left join tb_daily t2 
        on t1.data >= t2.data
            and t1.data - interval 28 days < t2.data
), tb_churn as 
(
-- se a pessoa está no próximo mau
select 
    t1.data ,
    t1.data + interval 28 day dt_churn, -- só observa o churn após 28 dias
    count(distinct t1.idcliente) as mau,
    count(distinct t2.idcliente) as mau_futuro,
    count(distinct t2.idcliente) / count(distinct t1.idcliente) tx_retencao,
    1 - (count(distinct t2.idcliente) / count(distinct t1.idcliente)) tx_churn
from tb_mau_users t1 
    left join tb_mau_users t2 
        on t1.data = t2.data - interval 28 day 
        and t1.idcliente = t2.idcliente
group by all
having count(distinct t2.idcliente) > 0
)
select 
    t1.*,
    t2.dt_churn,
    t2.tx_retencao,
    t2.tx_churn
from tb_wau t1 
    left join tb_churn t2 
        on t1.data = t2.data

Databricks visualization. Run in Databricks to view.

In [0]:
%sql

WITH tb_daily AS (

    SELECT DISTINCT
        DATE(DtCriacao) AS dtDia,
        IdCliente
    FROM tmw_loyalty.transacoes

),

tb_clientes AS (

    SELECT DISTINCT
        IdCliente
    FROM tb_daily

),

tb_datas AS (

    SELECT DISTINCT
        dtDia
    FROM tb_daily

),

tb_atividade AS (

    SELECT
        d.dtDia,
        c.IdCliente,
        MAX(t.dtDia) AS ultima_atividade

    FROM tb_datas d

    CROSS JOIN tb_clientes c

    LEFT JOIN tb_daily t
        ON t.IdCliente = c.IdCliente
        AND t.dtDia <= d.dtDia

    GROUP BY
        d.dtDia,
        c.IdCliente

),

tb_churn AS (

    SELECT
        dtDia,
        IdCliente,
        ultima_atividade,

        DATEDIFF(
            dtDia,
            ultima_atividade
        ) AS dias_inativo,

        CASE
            WHEN DATEDIFF(
                dtDia,
                ultima_atividade
            ) >= 28
            THEN 1
            ELSE 0
        END AS churn

    FROM tb_atividade

)

SELECT
    dtDia,
    COUNT(DISTINCT IdCliente) AS clientes,
    SUM(churn) AS clientes_churn,
    SUM(churn) / COUNT(DISTINCT IdCliente) AS taxa_churn

FROM tb_churn

GROUP BY dtDia
ORDER BY dtDia;

In [0]:
WITH datas AS (
    SELECT DISTINCT DATE(DtCriacao) AS Data
    FROM workspace.tmw_loyalty.transacoes
)

SELECT
    d.Data,
    COUNT(DISTINCT t.IdCliente) AS MAU
FROM datas d
JOIN workspace.tmw_loyalty.transacoes t
    ON DATE(t.DtCriacao) BETWEEN DATE_SUB(d.Data, 27) AND d.Data
GROUP BY d.Data
ORDER BY d.Data;